In [7]:
from tinygrad import Device
print(Device.DEFAULT)


METAL


In [8]:
from tinygrad import Tensor, nn 
class Model: 
    def __init__(self):
        self.l1 = nn.Conv2d(1, 32, kernel_size = (3, 3))
        self.l2 = nn.Conv2d(32, 64, kernel_size = (3, 3))
        # (H_in - kH + 2*padding)//stride + 1 (how to know output feature map's H and W)
        # 28 - 3 / 1 + 1  = 26 feature map size 
        # 26 - 3 +
        self.l3 = nn.Linear(1600, 10)
    def __call__(self, x:Tensor) -> Tensor: 
        x = self.l1(x).relu().max_pool2d((2, 2))
        x = self.l2(x).relu().max_pool2d((2, 2))
        return self.l3(x.flatten(1).dropout(0.5))
        

In [9]:
from tinygrad.nn.datasets import mnist
X_train, Y_train, X_test, Y_test = mnist()
print(X_train.shape, X_train.dtype, Y_train.shape, Y_train.dtype)


(60000, 1, 28, 28) dtypes.uchar (60000,) dtypes.uchar


In [10]:
model = Model() 
acc = (model(X_test).argmax(axis=1) == Y_test).mean()
print(acc.item())  # ~10% accuracy, as expected from a random model


0.11050000041723251


In [11]:
optim = nn.optim.Adam(nn.state.get_parameters(model)) 
batch_size = 128 
def step():
    Tensor.training = True 
    samples = Tensor.randint(batch_size, high=X_train.shape[0])
    X, Y = X_train[samples], Y_train[samples]
    optim.zero_grad() 
    loss = model(X).sparse_categorical_crossentropy(Y).backward()
    optim.step()
    return loss



In [15]:
import timeit
timeit.repeat(step, repeat=5, number=1)


[0.24058825001702644,
 0.2247867080150172,
 0.2269908750022296,
 0.22276754101039842,
 0.21897287500905804]

In [16]:
from tinygrad import GlobalCounters, Context
GlobalCounters.reset()
with Context(DEBUG=2): step()


scheduled 56 kernels in 180.41 ms
*** METAL      1 En11                                           arg  1 mem   0.06 GB tm      8.38us/     0.01ms (      0 GFLOPS    0|0      GB/s) ['__imul__']
*** METAL      2 En12                                           arg  1 mem   0.06 GB tm      8.21us/     0.02ms (      0 GFLOPS    0|0      GB/s) ['__imul__']
*** METAL      3 En7                                            arg  1 mem   0.06 GB tm      8.42us/     0.03ms (      0 GFLOPS    0|0      GB/s) ['randint']
*** METAL      4 E_32_4                                         arg  3 mem   0.06 GB tm     25.87us/     0.05ms (      1 GFLOPS    0|0      GB/s) ['randint']
*** METAL      5 En10                                           arg  2 mem   0.06 GB tm      6.75us/     0.06ms (      0 GFLOPS    0|0      GB/s) ['dropout']
*** METAL      6 E_32_49_5_4_16_2_4                             arg  3 mem   0.06 GB tm    175.00us/     0.23ms (      0 GFLOPS   46|46     GB/s) ['randint', '__getitem__']
*